# Tasks i-v ???? oder alle??

## Setup

**Load packages**


In [1]:
import numpy as np
from pathlib import Path
import pandas as pd

**Load data**

In [2]:
#load data
Code_folder= Path.cwd().parent
data = pd.read_csv(Code_folder / "Data" / "aapl_28apr.csv", header=2)
#rename columns and to date time format
data.columns=["Date", "Price level"]
data["Date"]= pd.to_datetime(data["Date"])
data.set_index("Date", inplace=True)
print(data)

            Price level
Date                   
2020-01-02    72.400505
2020-01-03    71.696655
2020-01-06    72.267921
2020-01-07    71.928062
2020-01-08    73.085106
...                 ...
2026-04-22   273.170013
2026-04-23   273.429993
2026-04-24   271.059998
2026-04-27   267.609985
2026-04-28   270.709991

[1588 rows x 1 columns]


**Define and derive variables and model parameters**

In [3]:
#define S_0 as the observation from 28.04.2026
S_0=data["Price level"].iloc[-1]
#make a log return column
data["Log return"]= np.log(data["Price level"] /data["Price level"].shift(1))
#use historical daily returns up to 27.04 2026 for volatility
returns_for_volatility=data.loc[: "2026-04-27", "Log return"].dropna()
#daily  volatility as standard deviation
sigma_daily=returns_for_volatility.std()
#annualize volatility
sigma = sigma_daily*np.sqrt(250)
print("annualized volatility equals", sigma)

#given model parameters
T= 0.5 #this means 6 months
n= 25
r= 0.01       
#time step
delta_t= T/n
#up and down factors
u= np.exp(sigma* np.sqrt(delta_t))
d=np.exp(-sigma*np.sqrt(delta_t))
#real probability, as specified in the exercise
p=0.5
#risk neutral probability


annualized volatility equals 0.313100818870145


## Task i: Build binomial tree


Build a binomial tree, first building the binomial tree using the method described in https://www.kaggle.com/code/ahmetgokkaya/pricing-derivatives-with-binomial-tree-model/notebook.

In [4]:
def binomial_tree(S_0, u, d, n):
    #initialize n+1 x n+1 matrix with entry [j, t] holding price at time t with
    #j down-moves (-> t-j up-moves)
    stock_tree=np.zeros((n+1, n+1))
    #set value of t=0 into matrix
    stock_tree[0, 0]= S_0
    #fill in the binomial tree like formula
    for i in range(1, n+1):
        for j in range(i+1):
            stock_tree[j, i]=S_0*u**(i-j)*d**j
   
    return stock_tree  


In [33]:
stock_tree = binomial_tree(S_0, u, d, n)

## Task ii: Calculate the arithmetic average stock price

**Function to capture sum (=path dependent)**:
Need to capture the fact that although at maturity time $T$ some $S_j$ will have the same payoff, they have gone through a different path and therefore have a different sum. We follow for this the idea of an augmented state space of Hull, J. and White, A. (1993). Efficient procedures for valuing European and American path-dependent options. Journal of Derivatives, 1(1), 21–31. but instead of interpolation we'll store every reachable sum using a dictionary. This will ease later computations.

We count (in the dictionary) how many paths lead to the same arithmetic average at the same terminal stock node because in binomial trees same average may be reached by more than one path. For option pricing later, this matters because the option value depends on the probability-weighted payoff.

We round entries to 10 decimals to avoid artificial duplicate states caused by tiny floating-point errors.

In [14]:
def Sum(S_0, u, d, n):
    #get matrix of the binomial tree prices
    stock_tree=binomial_tree(S_0, u, d, n)
    #initialize tree of dictionaries to capture the sum
    tree_dics=[[{} for k in range(t+1)] for t in range(n+1)]
    #for t=0 we already know the initial price which is also the sum at t=0
    tree_dics[0][0]= {round(S_0, 10): 1}
    #build sum dictionaries
    for t in range(n):
        for k in range(t+1):
            for total_sum, count in tree_dics[t][k].items():
                #up move-> get to node (k, t+1)
                S_up=stock_tree[k, t+1]
                #add S_up to the sum
                new_sum_up=round(total_sum + S_up, 10)
                tree_dics[t+1][k][new_sum_up]=tree_dics[t+1][k].get(new_sum_up, 0)+count
                
                #down move-> get to node (k+1, t+1)
                S_down=stock_tree[k+1, t+1]
                new_sum_down=round(total_sum + S_down, 10)
                tree_dics[t+1][k+1][new_sum_down]=tree_dics[t+1][k+1].get(new_sum_down, 0)+count

    return tree_dics

**Get the arithmetic mean**

In [15]:
def arithmetic_mean(S_0, u, d, n):
    #get dic of sum
    tree_dics= Sum(S_0, u, d, n)
    #initialize dictionary for the arithmetic average for each terminal node
    mean_dics= [{} for node in range(n+1)]
    #loop through nodes and get for all sum entries the arithmetic average
    for k in range(n+1):
        for total_sum, count in tree_dics[n][k].items():
            #arithmetic average
            avg=round(total_sum/(n+1), 10)
            mean_dics[k][avg]=mean_dics[k].get(avg, 0)+count
    
    return mean_dics

In [16]:
#print result all upmoves
print(arithmetic_mean(S_0, u, d, n)[0])

{np.float64(497.2450800392): 1}


In [34]:
tree_dics= Sum(S_0, u, d, n)
mean_dics= arithmetic_mean(S_0, u, d, n)

## Task iii: floating-strike Asian call option

In [31]:
def Asian_call(S_0, u, d, n):
    #get stock price matrix and sum dictionaries
    stock_tree= binomial_tree(S_0, u, d, n)
    tree_dics= Sum(S_0, u, d, n)    
    #initialize dictionary to store payoffs at each terminal node 
    payoff_dics=[{} for node in range(n+1)]    
    #go through all terminal nodes and calculate the payoff of call option
    for k in range(n+1):
        S_n=stock_tree[k, n]
        for total_sum, count in tree_dics[n][k].items():
            #round cumulative sum again to avoid artificial duplicate states caused by floating-point errors
            total_sum=round(total_sum, 10)
            #arithmetic average
            avg=round(total_sum/(n+1), 10)
            #floating-strike Asian call payoff
            payoff=max(S_n-avg, 0)
            #use total_sum as key because later tasks need the cumulative sum state
            payoff_dics[k][total_sum]={"S_n": S_n, "avg": avg, "payoff": payoff, "count": count}    
    return payoff_dics

In [32]:
#print call payoff from all up movement
asian_call=Asian_call(S_0, u, d, n)
print(asian_call[0])

{np.float64(12928.3720810195): {'S_n': np.float64(818.9529767238812), 'avg': np.float64(497.2450800392), 'payoff': np.float64(321.7078966846812), 'count': 1}}


In [36]:
payoff_dics = Asian_call(S_0, u, d, n)